# Lending Club Loan Performance & Risk Analysis (2007–2018)

Peer-to-peer lending platforms must carefully evaluate borrower risk while maintaining profitability. Poor credit risk assessment can lead to high default rates, reducing investor confidence and financial sustainability.

Lending companies need to understand which borrower characteristics, loan attributes, and economic factors are associated with higher default rates and lower loan performance.

Using historical lending data from Lending Club (2007-2018), this project analyzes loan performance to identify patterns in borrower behavior, loan characteristics, and repayment outcomes. The goal is to generate actionable insights that can help lending companies:

- Identify high-risk borrower profiles

- Understand the relationship between loan characteristics and default rates

- Evaluate the profitability of different loan segments

- Improve lending strategies and risk management decisions

The analysis uses Python for data exploration and cleaning, SQL for business queries, and Power BI to build an interactive dashboard for financial insights.

## Connecting dataset

I downloaded the full dataset but for knowledge purposes I'll simulate a SQL database extraction and different SQL queries.

In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("data/lending_analysis.db")

def run_query(query):
    return pd.read_sql(query, conn)

## Data Overview

Prior to any detailed analysis it’s good practice to inspect the raw table.  
In this section we

- report the **shape** (rows × columns).
- review each column’s **dtype**.
- compute basic **summary statistics** for numeric fields.

These checks provide a quick snapshot of data quality and help guide the cleaning and exploratory steps that follow.

### Shape

In [3]:
query = """
SELECT COUNT(*)
FROM loans;
"""

run_query(query)

,COUNT(*)
0,2260701


More than 2 millions of rows is a robust dataset to analyze and review, however, data may have missing or incorrect values.

### Dtype

In [4]:
query = """
PRAGMA table_info(loans);
"""

run_query(query)

,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,0
1,1,member_id,REAL,0,None,0
2,2,loan_amnt,REAL,0,None,0
3,3,funded_amnt,REAL,0,None,0
4,4,funded_amnt_inv,REAL,0,None,0
...,...,...,...,...,...,...
146,146,settlement_status,TEXT,0,None,0
147,147,settlement_date,TEXT,0,None,0
148,148,settlement_amount,REAL,0,None,0
149,149,settlement_percentage,REAL,0,None,0


Since the number of columns is 151, reducing it to the most important features is a good approach to focus the analysis in the most reliable columns.

### Summary statistics

In [6]:
query = """
SELECT 
    COUNT(*) AS total_loans,
    AVG(loan_amnt) AS avg_loan_amount,
    MIN(loan_amnt) AS min_loan_amount,
    MAX(loan_amnt) AS max_loan_amount,
    SUM(loan_amnt) AS total_loan_amount,
    AVG(int_rate) AS avg_interest_rate,
    MIN(int_rate) AS min_interest_rate,
    MAX(int_rate) AS max_interest_rate
FROM loans;
"""

run_query(query)

,total_loans,avg_loan_amount,min_loan_amount,max_loan_amount,total_loan_amount,avg_interest_rate,min_interest_rate,max_interest_rate
0,2260701,15046.931228,500.0,40000.0,3.401612e+10,13.092829,5.31,30.99


The summary statistics reveal key insights into the loan dataset:

- **Loan Amounts**: Average loan amount is around $15,000, with a range from $500 to $40,000, indicating diverse borrowing needs.
- **Interest Rates**: Average interest rate is approximately 13%, ranging from 5.32% to 30.99%, reflecting varying risk levels.
- **Loan Status Distribution**:
  - **Fully Paid (47.63%)**: Nearly half of loans are successfully repaid, showing strong performance in borrower selection.
  - **Current (38.85%)**: A significant portion are still active, representing ongoing risk and potential future repayments.
  - **Charged Off (11.88%)**: About 12% result in losses, highlighting the need for better risk assessment in this segment.
  - Other statuses (Late, Default) are minimal but indicate areas for monitoring.

These metrics suggest that while the platform has good repayment rates, there's room to reduce defaults through improved underwriting or monitoring strategies.

## Data Cleaning

Check for missing values, duplicates, and data type issues. Clean the data to ensure accuracy in analysis.

In [ ]:
query = """
SELECT *
FROM loans
LIMIT 5;
"""

pd.set_option("display.max_columns", None)
run_query(query)

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100
missing_summary = pd.DataFrame({'Missing Count': missing_values, 'Missing %': missing_percentage})
print("Missing values summary:")
missing_summary[missing_summary['Missing Count'] > 0].sort_values('Missing %', ascending=False)

# Check for duplicates
duplicates = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicates}")

# Data type conversions if needed (e.g., dates)
# df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], errors='coerce')

## Exploratory Data Analysis (EDA)

Explore the data through visualizations and statistical summaries to uncover patterns, relationships, and insights.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of loan amounts
plt.figure(figsize=(10, 6))
sns.histplot(df['loan_amnt'], bins=50, kde=True)
plt.title('Distribution of Loan Amounts')
plt.xlabel('Loan Amount ($)')
plt.ylabel('Frequency')
plt.show()

# Loan status by grade
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='grade', hue='loan_status', order=sorted(df['grade'].unique()))
plt.title('Loan Status by Grade')
plt.show()

# Correlation heatmap for numerical variables
numerical_cols = ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'fico_range_low', 'fico_range_high']
corr_matrix = df[numerical_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

## Key Insights and Recommendations

Summarize the main findings from the analysis and provide actionable recommendations for Lending Club.

### Recommendations for a Perfect Data Analyst Notebook

1. **Structure with Clear Sections**: Use markdown headers to organize the notebook into logical sections (e.g., Import, Overview, Cleaning, EDA, Insights).

2. **Document Everything**: Add comments in code cells and explanatory text in markdown cells to explain why you're doing each step and what the results mean.

3. **Handle Data Quality**: Always check for missing values, outliers, and data types. Decide on imputation or removal strategies based on business context.

4. **Use Visualizations**: Include charts and graphs to make insights visual and easier to understand. Label axes and add titles.

5. **Focus on Business Value**: Tie analysis back to the original questions (e.g., which customers are likely to default, profitability by segment).

6. **Ensure Reproducibility**: Include environment setup (library versions), and make sure the notebook runs from top to bottom.

7. **Summarize Key Findings**: End with a concise summary of insights and recommendations, avoiding technical jargon for non-technical audiences.

8. **Version Control**: Use Git to track changes, and consider exporting to PDF or HTML for sharing.

9. **Performance Considerations**: For large datasets, use sampling or efficient queries to avoid long run times.

10. **Ethical Considerations**: Be mindful of data privacy and bias in analysis, especially with financial data.

## Loan status distribution

In [4]:
query = """
SELECT 
    loan_status,
    COUNT(*) AS total_loans,
    ROUND(COUNT(*) * 100.0 / (
        SELECT COUNT(*) 
        FROM loans 
        WHERE loan_status IS NOT NULL
    ), 2) AS percentage
FROM loans
WHERE loan_status IS NOT NULL
GROUP BY loan_status
ORDER BY total_loans DESC;
"""

total_loans_by_status = run_query(query)
total_loans_by_status

,loan_status,total_loans,percentage
0,Fully Paid,1076751,47.63
1,Current,878317,38.85
2,Charged Off,268559,11.88
3,Late (31-120 days),21467,0.95
4,In Grace Period,8436,0.37
5,Late (16-30 days),4349,0.19
6,Does not meet the credit policy. Status:Fully ...,1988,0.09
7,Does not meet the credit policy. Status:Charge...,761,0.03
8,Default,40,0.00


1. **High Repayment Success**: Nearly half (47.63%) of all loans are fully paid, indicating that Lending Club's borrower selection and risk assessment are effective for a significant portion of applicants.

2. **Significant Ongoing Exposure**: Current loans account for 38.85% of the total, representing active accounts that could still transition to fully paid or charged off, requiring continuous monitoring.

3. **Moderate Loss Rate**: Charged-off loans make up 11.88% of the total, a notable but manageable portion of financial losses that highlights areas for improving default prevention.

## Average loan amount by loan status

In [6]:
query = """
SELECT 
    loan_status,
    AVG(loan_amnt) AS avg_loan_amount
FROM loans
GROUP BY loan_status
ORDER BY avg_loan_amount DESC;
"""

average_loan_amount_by_loan_status = run_query(query)
average_loan_amount_by_loan_status

,loan_status,avg_loan_amount
0,In Grace Period,17672.558084
1,Late (16-30 days),17391.118648
2,Late (31-120 days),16946.614571
3,Current,15942.815920
4,Charged Off,15565.055444
5,Default,14350.625000
6,Fully Paid,14134.369808
7,Does not meet the credit policy. Status:Charge...,9527.233903
8,Does not meet the credit policy. Status:Fully ...,8853.231891
9,NaN,NaN


In [6]:
query = """
SELECT
    -- Target variable
    loan_status,

    -- Credit profile
    fico_range_low, 
    fico_range_high, 
    grade, 
    sub_grade,
    delinq_2yrs, 
    inq_last_6mths,

    -- Debt & Income metrics
    dti, 
    annual_inc, 
    revol_util,

    -- Loan specifics
    loan_amnt, 
    int_rate, 
    term, 
    purpose,

    -- Credit history
    earliest_cr_line, 
    total_acc, 
    open_acc,
    mths_since_last_delinq, 
    mths_since_last_major_derog

FROM loans;
"""

df = run_query(query)